# Notebook 01: the original greedy agent
This is Part 2 of the assignment - running `main_work_greedy.py` on all levels and finding out why it fails.

Our simulator replays `CollectCoinThread.run()` from the original file. `create_matrix`, `check_coin`
and `findMinRoute` are copied as they are (we only removed the prints).

What the original does:
```
start_pos = [0.1, 0.1]
loop:
    read coin / monster / fire positions
    tsp   = create_matrix(start_pos, coins, monsters, fires)   # removes coins close to a hazard (< 0.3)
    route = findMinRoute(tsp)                                   # nearest neighbour route
    for each coin in route: walk to coin, sleep 0.4 s
```

In [ ]:
# mount google drive and make the folders we save everything into
import os, sys, json, subprocess, datetime, importlib
from google.colab import drive
drive.mount('/content/drive')

ROOT = '/content/drive/MyDrive/COMP6009_Assignment-Part A'   # all our files go here
PATHS = {
    'code':         f'{ROOT}/code',                          # the cointex_parta package
    'data':         f'{ROOT}/data',                          # cointex.kv, level table
    'config':       f'{ROOT}/config',                        # experiment_config.json
    'greedy':       f'{ROOT}/results/01_greedy_baseline',
    'proposed':     f'{ROOT}/results/02_proposed',
    'ga':           f'{ROOT}/results/03_ga_task2',
    'compare':      f'{ROOT}/results/04_comparison',
    'report_figs':  f'{ROOT}/report_assets/figures',         # final figures for the report/slides
    'report_tabs':  f'{ROOT}/report_assets/tables',          # final tables for the report
    'deliverables': f'{ROOT}/deliverables',
    'logs':         f'{ROOT}/logs',
}
for p in PATHS.values():
    os.makedirs(p, exist_ok=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pygad==3.3.1'], check=True)
if PATHS['code'] not in sys.path:
    sys.path.insert(0, PATHS['code'])

def log_run(notebook, **info):
    # keep a log of every run
    info.update(notebook=notebook, at=datetime.datetime.now().isoformat(timespec='seconds'))
    with open(f"{PATHS['logs']}/run_log.jsonl", 'a') as f:
        f.write(json.dumps(info) + '\n')

print('Project root:', ROOT)

In [ ]:
# load our code (written to drive by notebook 00) and the experiment settings
assert os.path.exists(f"{PATHS['code']}/cointex_parta/__init__.py"), \
    'code not found on Drive (notebook 00 writes it)'
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown
import cointex_parta
from cointex_parta import runner, plots, sim, levels
plots.style()
CFG = json.load(open(f"{PATHS['config']}/experiment_config.json"))
LEVELS, SEEDS = CFG['levels'], list(range(CFG['n_seeds']))
WORKERS = os.cpu_count()
print(f"cointex_parta v{cointex_parta.__version__} | {len(LEVELS)} levels x {len(SEEDS)} seeds | {WORKERS} CPU workers")
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)

for sub in ('figures', 'tables'): os.makedirs(f"{PATHS['greedy']}/{sub}", exist_ok=True)

## 1. Problems we found in the code
| # | Problem | Where | What happens |
|---|---|---|---|
| L1 | hazards only checked when the route is planned, not while walking it | `run()` | walks into moving monsters and fires |
| L2 | `check_coin` pops from the list while looping over it | `check_coin` | skips coins / removes the wrong ones, also uses corners not centres |
| L3 | the route always starts from (0.1, 0.1) | `run()` | after the first round the route is planned from the wrong place |
| L4 | no limit on move length (any distance in 0.25 s) | `start_char_animation` | one move can go straight through a hazard |
| L5 | sleeps 0.4 s per coin but the walk is 0.25 s | `run()` | wastes time every step |
| L6 | if all coins are near hazards the route is empty | `findMinRoute` | it just loops without moving |
| L7 | nearest neighbour only, no improvement | `findMinRoute` | longer routes |

Small examples of L2 and L3:

In [ ]:
from cointex_parta.agents import greedy as G
coins = [[0.10, 0.5], [0.20, 0.5], [0.30, 0.5], [0.40, 0.5], [0.80, 0.5]]
monster = [[0.25, 0.5]]
expected = [c for c in coins if abs(c[0]-0.25) + abs(c[1]-0.5) >= 0.3]
kept = G.check_coin([c[:] for c in coins], [m[:] for m in monster], [])
print('L2  coins that SHOULD survive (>=0.3 from monster):', expected)
print('    coins check_coin actually keeps            :', kept)

tsp = G.create_matrix([0.1, 0.1], [[0.9, 0.9], [0.8, 0.85]], [], [])
route = G.findMinRoute(tsp)
print('\nL3  character at (0.85,0.9) but tour planned from (0.1,0.1): route =', route,
      '-> visits', [[0.9, 0.9], [0.8, 0.85]][route[0]-2], 'first')

## 2. Running greedy on all 24 levels
Results are saved to `results/01_greedy_baseline/per_run.csv`.

In [ ]:
OUT = PATHS['greedy']
CSV = f'{OUT}/per_run.csv'
runner.run_sweep(CSV, LEVELS, SEEDS, ['greedy'], workers=WORKERS)
df = pd.read_csv(CSV)
log_run('01_greedy', episodes=len(df))
print(len(df), 'episodes')

In [ ]:
overall = runner.summarise(df)
per_level = runner.summarise(df, by=('agent', 'level'))
overall.to_csv(f'{OUT}/tables/overall_summary.csv', index=False)
per_level.to_csv(f'{OUT}/tables/per_level_summary.csv', index=False)
display(overall.round(2))
display(per_level[['level', 'success_rate', 'death_rate', 'timeout_rate', 'avg_coin_pct',
                   'completion_time_mean', 'remaining_life_mean', 'near_miss_pct_mean']].round(1))

## 3. Why does it fail?
For each lost episode we check whether the life was lost mostly to monsters or to fires.

In [ ]:
lost = df[df.won == 0].copy()
lost['cause'] = np.where(lost.timeout == 1, 'timeout',
                 np.where(lost.monster_hit_frames >= lost.fire_hit_frames, 'monster', 'fire'))
causes = lost.groupby(['level', 'cause']).size().unstack(fill_value=0)
causes.to_csv(f'{OUT}/tables/failure_causes_by_level.csv')
print('Failures by cause (all levels):'); print(lost.cause.value_counts().to_string())
print(f"\nMean coins collected before failing: {lost.coin_pct.mean():.1f}%  |  "
      f"mean time of failure: {lost.seconds.mean():.1f}s")
display(causes)

lv = list(range(1, 25))
c = causes.reindex(lv, fill_value=0)
fig, ax = plt.subplots(figsize=(13, 3.8))
bottom = np.zeros(len(lv))
for cause, col in [('monster', '#6250d6'), ('fire', '#e34948'), ('timeout', '#59636e')]:
    if cause in c:
        vals = 100 * c[cause].values / len(SEEDS)
        ax.bar(lv, vals, 0.7, bottom=bottom, color=col, label=cause, edgecolor='white', linewidth=0.6)
        bottom += vals
ax.set_xticks(lv); ax.set_xlabel('Level'); ax.set_ylabel('% of episodes lost')
ax.set_title('Original greedy: what kills it, by level', loc='left'); ax.legend(ncol=3)
ax.grid(axis='x', visible=False); fig.tight_layout()
plots.save(fig, f'{OUT}/figures', 'greedy_failure_causes.png'); plt.show()

In [ ]:
for metric, ylab, title, name, kw in [
        ('won', 'Success rate (%)', 'Original greedy: success rate by level', 'greedy_success_by_level.png', dict(scale=100, ylim=(0, 105))),
        ('completion_time', 'Seconds', 'Original greedy: completion time of won episodes', 'greedy_time_by_level.png', {}),
        ('remaining_life', 'Remaining life (%)', 'Original greedy: remaining life', 'greedy_life_by_level.png', dict(ylim=(0, 105)))]:
    fig = plots.per_level_bars(df, metric, ['greedy'], ylab, title, **kw)
    plots.save(fig, f'{OUT}/figures', name); plt.show()

## 4. Example of a failure
We replay the level where greedy does worst and save a picture and a gif (for the slides).

In [ ]:
worst_level = int(per_level.sort_values('success_rate').iloc[0]['level'])
lost = df[(df.level == worst_level) & (df.won == 0)]
seed = int(lost.seed.iloc[0]) if len(lost) else 0
row, trace = runner.play_episode(worst_level, seed, 'greedy', record=True)
print(row)
fig = plots.trajectory_grid({'greedy': trace}, ['greedy'], f'Original greedy on level {worst_level} (seed {seed})')
plots.save(fig, f'{OUT}/figures', f'greedy_failure_L{worst_level}_s{seed}.png'); plt.show()
gif = plots.animate_episode(trace, 'greedy', f'{OUT}/figures/greedy_failure_L{worst_level}_s{seed}.gif')
display(Image(filename=gif))

In [ ]:
headline = {
    'agent': 'greedy', 'episodes': int(len(df)),
    'success_rate_pct': round(100 * df.won.mean(), 2),
    'death_rate_pct': round(100 * df.died.mean(), 2),
    'timeout_rate_pct': round(100 * df.timeout.mean(), 2),
    'levels_always_won': int((df.groupby('level').won.mean() == 1).sum()),
    'completion_time_mean_s': round(df.completion_time.mean(), 2),
    'remaining_life_mean_pct': round(df.remaining_life.mean(), 2),
    'worst_levels': per_level.sort_values('success_rate').head(5)[['level', 'success_rate']].to_dict('records'),
}
json.dump(headline, open(f'{OUT}/headline.json', 'w'), indent=2)
headline